## Student Attendance Analytics with Pandas

In [1]:
import pandas as pd
import numpy as np

#### Section 0: Data Loading

In [ ]:
df = pd.read_excel("subdata_pandas/CSEB_Attendance.xlsx")
df

#### Section 1: Data Inspection

In [ ]:
"""Q1. Display the first 10 rows, the shape of the DataFrame, and the data types of each column."""
df.head(10)

df.shape

df.dtypes

In [ ]:
"""Q2. Check for missing values in each column. Identify which columns have null values (especially Last Present Date with "Never") and report the count. """
df.isna()
df.isna().sum()



In [ ]:
"""Q3. Rename the columns to snake_case (e.g., Roll No → roll_no, Student Name → student_name, Attendance % → attendance_pct). Display the updated column list."""
df.rename(columns={
    "Roll No": "roll_no",
    "Student Name": "student_name",
    "Attendance %": "attendance_pct"
}, inplace=True)
df

In [ ]:
"""Q4. Convert Attendance % from string format (e.g., "64%") to a numeric float column (0.64 or 64.0). Also convert Last Present Date to datetime, handling "Never" as NaT."""
# Convert Attendance % from string to float ("64%" -> 0.64)
raw_attendance = df["attendance_pct"]
has_percent_sign = raw_attendance.astype("string").str.contains("%", na=False)

attendance_values = pd.to_numeric(
    raw_attendance.astype("string").str.replace("%", "", regex=False),
    errors="coerce"
)

# Convert percentage strings or numeric values above 1; leave normalized values unchanged.
df["attendance_pct"] = attendance_values.where(
    ~(has_percent_sign | (attendance_values > 1)),
    attendance_values / 100
)

# Convert Last Present Date to datetime, handling "Never" as NaT
df["last_present_date"] = pd.to_datetime(
    df["Last Present Date"].replace("Never", pd.NaT),
    errors="coerce"
)

# Display the converted columns
display(df[["attendance_pct", "last_present_date"]].head())

#### Section 2: Filtering & Selection

In [48]:
"""Q5. Filter and display all students whose Status is "Critical". Sort them by Attendance % in ascending order."""
critical_students = df[df["Status"] == "Critical"].sort_values(
    by="attendance_pct",
    ascending=True
)

In [ ]:
"""Q6. Extract all students who have 0 present days (fully absent). List their roll numbers, names, and consecutive absences."""
fully_absent = df[df["Present"] == 0]

# Display roll number, name, and consecutive absences
display(fully_absent[[
    "roll_no",
    "student_name",
    "Consecutive Absences"
]])

In [ ]:
"""Q7. Using .loc and .iloc, display:
(a) All "Safe" students using .loc
(b) The first 15 rows and columns 2-6 using .iloc """

df.loc[df["Status"] == "Safe"]
df.iloc[:15, 1:6]

In [ ]:
"""Q8. Find all students whose Consecutive Absences is greater than or equal to 5. Display their name, status, and last present date."""
absent_students = df[df["Consecutive Absences"] >= 5]

# Display name, status, and last present date
display(absent_students[[
    "student_name",
    "Status",
    "Last Present Date"
]])

### Section 3: Grouping & Aggregation 

In [ ]:
"""Q9. Group the data by Status and compute: 
a) Count of students 
b) Average attendance %  
c) Maximum consecutive absences"""
# Group by Status and calculate the required statistics
summary = df.groupby("Status").agg(
    student_count=("Status", "count"),
    average_attendance=("attendance_pct", "mean"),
    max_consecutive_absences=("Consecutive Absences", "max")
)

display(summary)

In [ ]:
"""Q10. Create a new column Attendance Category:
≥ 90% → "Excellent"
75-89% → "Good"
60-74% → "Average"
< 60% → "Poor"
Then use groupby() to count students in each category."""

conditions = [
    df["attendance_pct"] >= 0.90,
    (df["attendance_pct"] >= 0.75) & (df["attendance_pct"] < 0.90),
    (df["attendance_pct"] >= 0.60) & (df["attendance_pct"] < 0.75),
    df["attendance_pct"] < 0.60
]

# Define category names
categories = ["Excellent", "Good", "Average", "Poor"]

# Create the new column
df["Attendance Category"] = np.select(conditions, categories, default="Unknown")

# Count students in each category
display(df.groupby("Attendance Category").size())

In [ ]:
"""Q11. Calculate the total number of classes conducted (sum of Total Classes) and the total present/absent counts across all students using .agg()."""
df[["Total Classes", "Present", "Absent"]].agg("sum")

In [ ]:
"""Q12. Group by Status and use .agg() to find:
a)min, max, and mean of Present
b)sum of Absent"""
# Group by Status and calculate the required statistics
result = df.groupby("Status").agg(
    Present_min=("Present", "min"),
    Present_max=("Present", "max"),
    Present_mean=("Present", "mean"),
    Absent_sum=("Absent", "sum")
)

display(result)

### Section 4: Feature Engineering & Transformation

In [ ]:
"""Q13. Create a new column Risk Score using the formula:
Risk Score = (Absent * 0.4) + (Consecutive Absences * 0.6)
Display the top 10 students with the highest risk score."""
# Create Risk Score column
df["Risk Score"] = (
    df["Absent"] * 0.4
    + df["Consecutive Absences"] * 0.6
)

# Display the top 10 students with the highest risk score
display(
    df.sort_values(by="Risk Score", ascending=False)
      .head(10)
)

In [ ]:
"""Q14. Add a column Classes Needed (Numeric) by extracting the numeric value from the Classes Needed (75%) column (e.g., "5 more" → 5, "✓ Met" → 0)."""
# Extract the numeric value from Classes Needed (75%)
df["Classes Needed (Numeric)"] = (
    df["Classes Needed (75%)"]
    .str.extract(r"(\d+)")
    .fillna(0)
    .astype(int)
)

display(df[["Classes Needed (75%)", "Classes Needed (Numeric)"]].head())

In [ ]:
"""Q15. Using apply() with a lambda function, create a column Attendance Flag that returns:
a)"Defaulter" if attendance < 75%
b)"Safe" otherwise """
# Create Attendance Flag using apply() and lambda
df["Attendance Flag"] = df["attendance_pct"].apply(
    lambda x: "Defaulter" if x < 0.75 else "Safe"
)

display(df[["attendance_pct", "Attendance Flag"]].head())

In [ ]:
"""Q16. Use pd.cut() to bin students into attendance ranges: [0-50, 50-75, 75-90, 90-100] and label them as ["Critical", "Warning", "Good", "Excellent"]."""
bins = [0, 50, 75, 90, 100]
labels = ["Critical", "Warning", "Good", "Excellent"]

# Create attendance ranges
df["Attendance Range"] = pd.cut(
    df["attendance_pct"] * 100,
    bins=bins,
    labels=labels,
    include_lowest=True
)
df

### Section 5: Advanced Analysis 

In [ ]:
"""Q17. Using pivot_table(), create a summary showing the count of students by Status and Attendance Category (created in Q10)."""
# Create a pivot table showing student counts
summary = pd.pivot_table(
    df,
    index="Status",
    columns="Attendance Category",
    aggfunc="size",
    fill_value=0
)

display(summary)

In [ ]:
"""Q18. Identify duplicate student names (e.g., "ANANYA", "DISHA") and display all rows where the base name repeats. Use string manipulation to extract the base name."""
# Extract the base name (remove extra spaces and trailing numbers/suffixes if present)
df["base_name"] = (
    df["student_name"]
    .str.strip()
    .str.extract(r"^([A-Za-z]+)")[0]
    .str.upper()
)

# Identify all rows where the base name repeats
duplicates = df[
    df["base_name"].duplicated(keep=False)
].sort_values(by="base_name")

df

In [ ]:
""""Q19. Merge this dataset with a hypothetical Student_Details DataFrame containing Roll No, Branch, and Semester. Perform an inner join and display the merged result."""
student_details = pd.DataFrame({
    "Roll No": [101, 102, 103, 104, 105],
    "Branch": ["CSE", "ISE", "CSE", "ECE", "AIML"],
    "Semester": [5, 5, 5, 5, 5]
})

# Perform an inner join
student_details = student_details.rename(columns={"Roll No": "roll_no"})

student_details["roll_no"] = df["roll_no"].head(len(student_details)).to_numpy()

merged_df = pd.merge(
    df,
    student_details,
    on="roll_no",
    how="inner"
)

display(merged_df)


In [63]:
"""Q20. Export the following to separate CSV files:
(a) All "Critical" students
(b) Summary statistics grouped by Status
(c) Top 10 students by Risk Score"""
#a
critical_students = df[df["Status"] == "Critical"]
critical_students.to_csv("critical_students.csv", index=False)

#b
status_summary = df.groupby("Status").agg({
    "Present": ["min", "max", "mean"],
    "Absent": "sum",
    "Total Classes": "sum"
})
status_summary.to_csv("status_summary.csv")

#c
top_10_risk = df.sort_values(
    by="Risk Score",
    ascending=False
).head(10)

top_10_risk.to_csv("top_10_risk_score.csv", index=False)